### Importing data set

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df= pd.read_csv('https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/main/day28-column-transformer/covid_toy.csv')

df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [2]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

### Train Test Split

In [8]:
X=df.drop('has_covid' , axis=1)
Y=df['has_covid']

X_train, X_test, Y_train, Y_test= train_test_split(X, Y, test_size=0.2, random_state=42)

### Feature Scaling

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [13]:
transformer= ColumnTransformer(transformers=[
    
    # Missing Fever values 
    ('fever_imputer', SimpleImputer(), ['fever']),

    # Cough (Ordinal Encoding)
    ('cough_ordinal', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),

    # Gender, City (One Hot Encoding)
    ('gender_city_ohe', OneHotEncoder(drop='first', sparse_output=False), ['gender','city']),
], remainder='passthrough')

### Transform Data

In [14]:
X_train_transformed= transformer.fit_transform(X_train)

X_test_transformed= transformer.transform(X_test)

### Scaling and transforming Y-axis

In [17]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

le=LabelEncoder()

Y_train_encoded=le.fit_transform(Y_train)
Y_test_encoded=le.transform(Y_test)

### Making Pipeline

In [20]:
from sklearn.pipeline import Pipeline

pipe=Pipeline([
    ('preprocessing', transformer)
])

pipe.fit(X_train)

X_train_final=pipe.transform(X_train)
X_test_final=pipe.transform(X_test)

### Output

In [21]:
pd.DataFrame(X_train_final).head()

,0,1,2,3,4,5,6
0,101.0,0.0,0.0,0.0,0.0,1.0,81.0
1,100.0,0.0,0.0,0.0,1.0,0.0,5.0
2,100.0,0.0,0.0,0.0,1.0,0.0,19.0
3,100.0,0.0,1.0,1.0,0.0,0.0,27.0
4,103.0,0.0,0.0,1.0,0.0,0.0,73.0


In [22]:
# 1. Get the names of the columns from the transformer inside the pipe
new_columns = pipe.named_steps['preprocessing'].get_feature_names_out()

# 2. Create a DataFrame using the transformed data and the new names
X_train_final_df = pd.DataFrame(X_train_final, columns=new_columns)

# 3. Look at the result
X_train_final_df.head()

,fever_imputer__fever,cough_ordinal__cough,gender_city_ohe__gender_Male,gender_city_ohe__city_Delhi,gender_city_ohe__city_Kolkata,gender_city_ohe__city_Mumbai,remainder__age
0,101.0,0.0,0.0,0.0,0.0,1.0,81.0
1,100.0,0.0,0.0,0.0,1.0,0.0,5.0
2,100.0,0.0,0.0,0.0,1.0,0.0,19.0
3,100.0,0.0,1.0,1.0,0.0,0.0,27.0
4,103.0,0.0,0.0,1.0,0.0,0.0,73.0
